# Day 2 · SciPy

**Block:** SciPy (25 min, including slides and live coding)

1. **Curve fitting** (8 min): after SPL Exercise 39, fit the mouse's
   psychometric curve in each block of trials
2. **Filtering with the FFT** (8 min): after SPL Exercise 42, remove the mains
   hum from the LFP
3. **Stretch:** which distribution fits the response times (after SPL
   Exercise 41), 2-D minimisation (SPL Exercise 40), then the chapter's
   summary exercises

We keep using the recording from Day 1. The exercises are adapted from
[Scientific Python Lectures, SciPy: high-level scientific
computing](https://lectures.scientific-python.org/intro/scipy/index.html) (CC
BY 4.0), which fits Alaska's temperatures and cleans up a photograph instead.

Before writing your own algorithm, **check whether SciPy already has it**.
The [API reference](https://docs.scipy.org/doc/scipy/reference/) lists every submodule.

In [ ]:
import h5py
import matplotlib.pyplot as plt
import numpy as np
import scipy as sp

with h5py.File("../data/ibl_session.h5") as f:
    contrast = f["trials/contrast"][:]
    choice = f["trials/choice"][:]
    probability_left = f["trials/probability_left"][:]
    rt = f["trials/response_time_s"][:]
    lfp_uv = f["lfp/data"][:] * f["lfp"].attrs["uV_per_count"]
    sampling_rate = f["lfp"].attrs["sampling_rate_Hz"]

## 1. Curve fitting (after SPL Exercise 39)

On Day 1 you computed the **psychometric curve**: the fraction of rightward
choices at each contrast. The task switches between blocks in which the
stimulus is more often on the left (`probability_left` 0.8) or on the right
(0.2). Here is the curve for each kind of block:

In [ ]:
levels = np.unique(contrast)
right_block = probability_left == 0.2
left_block = probability_left == 0.8
frac_right_rb = np.array([(choice[right_block & (contrast == c)] == 1).mean() for c in levels])
frac_right_lb = np.array([(choice[left_block & (contrast == c)] == 1).mean() for c in levels])
levels, frac_right_rb.round(2), frac_right_lb.round(2)

1. Plot both curves against contrast.
2. Write a function `psychometric(c, bias, width, lapse)` that can describe
   both. A common choice is a cumulative Gaussian, which rises from `lapse` to
   `1 - lapse`, is centred on `bias` and rises over about `width` (in %
   contrast):

   $$p(c) = \text{lapse} + (1 - 2\,\text{lapse})\, \tfrac{1}{2}\left(1 + \text{erf}\!\left(\frac{c - \text{bias}}{\sqrt{2}\,\text{width}}\right)\right)$$

   `sp.special.erf` is the error function.
3. Fit it to each curve with `sp.optimize.curve_fit`, starting from
   `p0=[0, 10, 0.05]`.
4. Plot the fits on a fine contrast grid. Is the fit reasonable?
5. Is the `bias` the same in the two kinds of block, within the fit's
   accuracy? (Hint: `curve_fit` also returns a covariance matrix. The square
   roots of its diagonal are the parameters' standard errors.) What does the
   answer say about the mouse?

In [ ]:
# Your code here


In [ ]:
# Your code here


*Your answer here.*

## 2. Filtering with the FFT (after SPL Exercise 42)

The recording was made in the United States, where mains electricity
alternates at **60 Hz**. Every cable in the room radiates it, and some ends up
in the LFP. Remove it with the Fast Fourier Transform:

1. Take `trace`, channel 200 of `lfp_uv`, and plot its first 0.5 s.
2. Find the FFT for real signals in `sp.fft` (`rfft`), and the matching
   frequencies (`rfftfreq`, which needs the time between samples). Plot the
   spectrum (the absolute value) against frequency. Is it hard to see
   anything? Why? (Hint: `ax.set_yscale("log")`.)
3. Find the mains hum in the spectrum. Set the components within 1 Hz of
   60 Hz **and of its harmonics**, 120 and 180 Hz, to zero, using a mask on the
   frequencies.
4. Apply the inverse FFT (`irfft`, with `n=` the length of `trace`), and plot
   what you **removed**, `trace - cleaned`, for the first 0.2 s. What does it
   look like?

In [ ]:
trace = lfp_uv[:, 200]
time = np.arange(trace.size) / sampling_rate

In [ ]:
# Your code here


In [ ]:
# Your code here


*Your answer here.*

In [ ]:
# Your code here


## 3. Stretch

### Which distribution fits the response times? (after SPL Exercise 41)

Response times are positive and skewed, with a long tail. Take the response
times below 5 s (the others are trials where the mouse was not engaged). Fit
a gamma distribution and a lognormal distribution to them with
`sp.stats.gamma.fit` and `sp.stats.lognorm.fit`, fixing the location at 0
(`floc=0`). Plot their PDFs on top of the histogram. Which fits better? Compare
the total log-likelihood, `dist.logpdf(sample, *params).sum()`, too: higher is
better.

Extra: plot the cumulative distribution function of the better fit, and use
its `.ppf` to find the response time that 90% of trials beat.

In [ ]:
# Your code here


In [ ]:
def sixhump(x):
    return (
        (4 - 2.1 * x[0] ** 2 + x[0] ** 4 / 3) * x[0] ** 2
        + x[0] * x[1]
        + (-4 + 4 * x[1] ** 2) * x[1] ** 2
    )

In [ ]:
# Your code here
